In [13]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [14]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    for item in result.get('data', []):
        if 'summary' in item:
            normalized_data.append(item)

observed_src = pd.json_normalize(normalized_data)
observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
observed_src.drop_duplicates(subset='indicator', inplace=True)
observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]


Querying owner: HTOC Org


In [15]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,ip,legacyLink,tags.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,4665818,2024-05-22T16:36:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-07T15:20:43Z,3.0,12,ESET researchers discovered two previously unk...,...,176.57.150.252,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 266153, 'name': 'Lua scripts', 'lastUs...",NaN,NaN,NaN,NaN,NaN,NaN,176.57.150.252
1,6755399460010853,2025-07-03T14:03:25Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-07T15:20:29Z,3.0,90,This indicator is associated with a malspam ca...,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1456149, 'name': 'hhs-2025070301', 'la...",pdf.ac,False,False,NaN,NaN,NaN,pdf.ac
2,4697046,2024-06-05T13:07:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-07T14:11:42Z,5.0,90,"Starting in April 2024, UNC5537 used stolen cr...",...,194.230.144.126,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 267649, 'name': 'Snowflake', 'lastUsed...",NaN,NaN,NaN,NaN,NaN,NaN,194.230.144.126
3,6755399460007548,2025-07-02T12:05:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-07T14:11:38Z,4.0,73,"Details\nIn late June 2025, Iran-aligned hackt...",...,195.47.238.91,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1455870, 'name': 'Mr Hamza Group', 'la...",NaN,NaN,NaN,NaN,NaN,NaN,195.47.238.91
4,6755399460007553,2025-07-02T12:05:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-07T14:11:29Z,4.0,73,"Details\nIn late June 2025, Iran-aligned hackt...",...,198.98.62.158,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1455870, 'name': 'Mr Hamza Group', 'la...",NaN,NaN,NaN,NaN,NaN,NaN,198.98.62.158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1097,4320234,2023-04-13T12:59:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-02-11T23:34:01Z,1.0,5,"On Wednesday, April 4, 2023, VA users received...",...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,http://click.email.active.com,NaN,click.email.active.com,click.email.active.com
1099,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90,ACD R&F processed a malspam campaign with a Ne...,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 390199, 'name': 'hhs-2024090901', 'las...",NaN,NaN,NaN,https://www.shorturl.at/,NaN,www.shorturl.at/,www.shorturl.at/
1100,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,http://selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com,selligenttier.naylorcampaigns.com
1102,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,https://google,NaN,google,google


In [16]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250707.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250706.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250705.csv']
Loaded data from 3 files.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_16460\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


In [17]:
import warnings

# Extract API-tagged indicators
all_filtered = []
cutoff = pd.Timestamp.utcnow()

for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if isinstance(tags_data, list):
        tags_df = pd.json_normalize(tags_data)
        api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)]
        if not api_tags.empty:
            all_tags_list = tags_df['name'].astype(str).tolist()
            api_tags['name'] = api_tags['name'].astype(str)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink','rating', 'confidence', 'id']:
                api_tags[col] = row.get(col)
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            all_filtered.append(api_tags)
            warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
filtered_tags = pd.concat(all_filtered, ignore_index=True)
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Prepare for matching
filtered_tags['OpDiv'] = filtered_tags['name'].str.replace(' Splunk API', '', regex=False)
obs_subset = observed_data_df[['indicator', 'OpDiv', 'obs_date']].drop_duplicates()
recent_tags = pd.merge(filtered_tags, obs_subset, left_on=['summary', 'OpDiv'], right_on=['indicator', 'OpDiv'], how='inner')

# Filter to recent
cutoff_naive = cutoff.tz_convert(None)
recent_tags = recent_tags[
    (recent_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (pd.to_datetime(recent_tags['obs_date'], errors='coerce') >= cutoff_naive - timedelta(days=1))
].copy()

# Partner processing
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
recent_tags.drop_duplicates(subset='summary', inplace=True)


In [18]:
# Enrich only final filtered indicators
indicator_values = recent_tags['summary'].dropna().unique().tolist()
enriched_results = []

print(f"Enriching {len(indicator_values)} indicators with DomainTools and VirusTotalV3...")

for value in indicator_values:
    try:
        # Use the indicator *value*, not the ID
        encoded_value = urllib.parse.quote(value)
        enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
        ro.set_http_method('POST')
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        enrich_response = tc.api_request(ro)

        if enrich_response.status_code == 200:
            enrich_data = enrich_response.json()
            enrich_data['summary'] = value  # Save the value as key
            enriched_results.append(enrich_data)
    except Exception as e:
        continue

if enriched_results:
    df_enriched = pd.json_normalize(enriched_results)
    df_enriched = pd.json_normalize(enriched_results)
    recent_tags = recent_tags.merge(df_enriched, on='summary', how='left')
    # Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
else:
    print("No enrichment data retrieved.")


Enriching 597 indicators with DomainTools and VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host facebookmail.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host cdcfoundation.org cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host mavink.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host link.mail.beehiiv.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host com.ar cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"err

Successfully enriched and merged 576 indicators.


In [20]:
import numpy as np
import pandas as pd
from pandas import json_normalize

# Example data for `recent_tags` (replace with actual data loading)
recent_tags = pd.DataFrame({
    'indicator': ['1.1.1.1', '2.2.2.2'],
    'data.enrichment.data': [
        [{'type': 'Shodan', 'hostNames': ['host1'], 'domains': ['domain1'], 'tags': ['tag1'], 'country': 'US', 'city': 'New York', 'isp': 'ISP1', 'asn': 'AS123', 'org': 'Org1', 'openPorts': [{'port': 80, 'transport': 'tcp', 'product': 'HTTP'}]}],
        [{'type': 'VirusTotal', 'vtMaliciousCount': 5}]
    ],
    'obs_date': ['2025-07-07', '2025-07-08'],
    'rating': ['High', 'Low'],
    'confidence': ['High', 'Low'],
    'partners': ['Partner1', 'Partner2'],
    'shodan_openPorts': [None, None]
})

def extract_enrichment(row):
    """Extracts enrichment fields from the list of dicts in 'data.enrichment.data'."""
    enrichments = row.get('data.enrichment.data', [])
    result = {}
    
    for enrich in enrichments:
        enrich_type = enrich.get('type')
        if enrich_type == 'Shodan':
            for key in ['hostNames', 'domains', 'tags', 'country', 'city', 'isp', 'asn', 'org', 'openPorts']:
                result[f'shodan_{key}'] = enrich.get(key, np.nan)
        elif enrich_type == 'VirusTotal':
            result['vtMaliciousCount'] = enrich.get('vtMaliciousCount', np.nan)
    
    return pd.Series(result)

# Apply extraction to recent_tags
enrichment_expanded = recent_tags.apply(extract_enrichment, axis=1)

def unnest_ports(ports):
    """Unnest the Ports column to extract detailed information."""
    if isinstance(ports, list):
        unnested = []
        for port_info in ports:
            port_details = {
                'transport': port_info.get('transport', 'unknown'),
                'port': port_info.get('port', 'unknown'),
                'product': port_info.get('product', 'unknown'),
                'data': port_info.get('data', 'unknown'),
                'ssl_issuer': port_info.get('ssl', {}).get('issuer', 'unknown'),
                'ssl_subject': port_info.get('ssl', {}).get('subject', 'unknown'),
                'ssl_issued': port_info.get('ssl', {}).get('issued', 'unknown'),
                'ssl_expires': port_info.get('ssl', {}).get('expires', 'unknown')
            }
            unnested.append(port_details)
        return unnested
    return np.nan

# Expand ports details if 'Ports' exists
if 'Ports' in recent_tags.columns:
    recent_tags['Ports_Details'] = recent_tags['Ports'].apply(unnest_ports)

# Flatten the 'Ports_Details' into separate columns
if 'Ports_Details' in recent_tags.columns:
    recent_tags = recent_tags.explode('Ports_Details')
    ports_normalized = json_normalize(recent_tags['Ports_Details'])
    recent_tags = pd.concat([recent_tags.drop(columns=['Ports_Details']), ports_normalized], axis=1)

# Merge enriched data
recent_tags_expanded = pd.concat([recent_tags, enrichment_expanded], axis=1)

# Rename columns for clarity
recent_tags_expanded = recent_tags_expanded.rename(columns={
    'indicator': 'Indicator',
    'vtMaliciousCount': 'Malicious Score/Count',
    'obs_date': 'Observation Date',
    'shodan_asn': 'ASN',
    'rating': 'ThreatAssessRating',
    'confidence': 'ThreatAssessConfidence',
    'shodan_city': 'City',
    'shodan_country': 'Country',
    'data.legacyLink': 'ThreatConnect Link',
    'partners': 'Partners',
    'shodan_openPorts': 'Ports'
})

# Select only the desired columns
desired_columns = [
    'Indicator',
    'Malicious Score/Count',
    'Observation Date',
    'ASN',
    'ThreatAssessRating',
    'ThreatAssessConfidence',
    'City',
    'Country',
    'ThreatConnect Link',
    'Partners',
    'Ports'
]

# Handle missing 'Ports' and 'shodan_openPorts' fields
if 'Ports' not in recent_tags_expanded.columns or recent_tags_expanded['Ports'].isnull().any():
    recent_tags_expanded['Ports'] = recent_tags_expanded.get('shodan_openPorts', 'unknown')

# Keep only existing columns from desired_columns
existing_columns = [col for col in desired_columns if col in recent_tags_expanded.columns]
recent_tags_expanded = recent_tags_expanded[existing_columns]

# Drop duplicate columns and fill missing values
recent_tags_expanded = recent_tags_expanded.loc[:, ~recent_tags_expanded.columns.duplicated()]
recent_tags_expanded = recent_tags_expanded.fillna('unknown')

# Ensure Ports column is formatted properly
if 'Ports' in recent_tags_expanded.columns:
    recent_tags_expanded['Ports'] = recent_tags_expanded['Ports'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Remove duplicate rows
recent_tags_expanded = recent_tags_expanded.drop_duplicates()

print(recent_tags_expanded)


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
enrichment_expanded

,shodan_asn,shodan_city,shodan_country,shodan_domains,shodan_hostNames,shodan_isp,shodan_openPorts,shodan_org,shodan_tags,vtMaliciousCount
0,AS396527,Cambridge,United States,NaN,NaN,Massachusetts Institute of Technology,"[{'transport': 'tcp', 'port': 2222, 'product':...",MIT,[tor],8.0
0,AS396527,Cambridge,United States,NaN,NaN,Massachusetts Institute of Technology,"[{'transport': 'tcp', 'port': 2222, 'product':...",MIT,[tor],8.0
1,AS30633,Washington,United States,NaN,NaN,"Leaseweb USA, Inc.","[{'transport': 'tcp', 'port': 80, 'product': '...","Leaseweb USA, Inc.",[tor],6.0
1,AS30633,Washington,United States,NaN,NaN,"Leaseweb USA, Inc.","[{'transport': 'tcp', 'port': 80, 'product': '...","Leaseweb USA, Inc.",[tor],6.0
2,AS64289,Amsterdam,Netherlands,NaN,NaN,Macarne.com,"[{'transport': 'tcp', 'port': 22, 'product': '...",Macarne Limited,NaN,5.0
...,...,...,...,...,...,...,...,...,...,...
593,AS20940,Ashburn,United States,"[akamaitechnologies.com, akamai.net]","[a248.e.akamai.net, a23-205-105-180.deploy.sta...",Akamai International B.V.,"[{'transport': 'tcp', 'port': 80, 'product': '...","Akamai Technologies, Inc.",[cdn],0.0
593,AS20940,Ashburn,United States,"[akamaitechnologies.com, akamai.net]","[a248.e.akamai.net, a23-205-105-180.deploy.sta...",Akamai International B.V.,"[{'transport': 'tcp', 'port': 80, 'product': '...","Akamai Technologies, Inc.",[cdn],0.0
594,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
595,AS396982,Kansas City,United States,"[ipv6addr.com, onlyip6.me, ip6.me, whatismyv6....","[145.111.160.34.bc.googleusercontent.com, ipv6...",Google LLC,"[{'transport': 'tcp', 'port': 80, 'data': 'HTT...",Google LLC,[cloud],2.0
